# 第0章 動作確認

環境構築マニュアルで用意した道具が、本当に動くのかを確かめる章です。ここで確かめることは3つだけです。ライブラリがそろっているか、計算に使う装置は CPU か GPU か、そして強化学習の「環境」が画面の絵を出せるか。最後に、棒を立てる課題（CartPole）を、何も考えずにでたらめに動かしてみます。棒はすぐに倒れてしまいますが、それで構いません。次の第1章では、同じ課題を学習させて、棒を立て続けられるようにします。

- **前提**: [環境構築マニュアル](../../setup/windows_setup.md)を終えていること
- **所要の目安**: 5分
- **使う装置**: CPU だけで動きます

> **注記**
> - **出力の出どころ**: この Notebook に残っている出力は、著者の環境（Windows 11、CPU のみ）で 2026-10-01 に実行したものです。各セルの出力がそのまま期待する結果で、その直後の「期待する結果」の段落に読み方を書いています。GPU がある環境では、装置の表示が変わります。
> - **進め方**: セルを上から順に実行します（VS Code では各セルの左の実行ボタン、または `Shift`+`Enter`）。

## 0. 学習目標と完了条件

この章を終えると、次のことができるようになります。

1. 教科書で使うライブラリの版を表示して確かめられる
2. 計算に使う装置（CPU / GPU）を確かめ、後の章で使う装置を選ぶ1行の書き方が分かる
3. Gymnasium の環境を動かし、その様子を GIF に保存して見られる

完了条件は、この Notebook を上から最後まで実行して、エラーが出ずに4節の GIF が表示されることです。

## 1. ライブラリの版を確かめる

教科書で使う主なライブラリを読み込み、版を表示します。`__version__` は、多くの Python のライブラリが持っている、版の番号を表す属性です。

In [1]:
import sys

import gymnasium as gym
import imageio
import matplotlib
import numpy as np
import stable_baselines3
import torch

print("Python           ", sys.version.split()[0])
print("Gymnasium        ", gym.__version__)
print("Stable-Baselines3", stable_baselines3.__version__)
print("PyTorch          ", torch.__version__)
print("NumPy            ", np.__version__)
print("matplotlib       ", matplotlib.__version__)
print("imageio          ", imageio.__version__)

Python            3.13.16
Gymnasium         1.3.0
Stable-Baselines3 2.9.0
PyTorch           2.14.1+cpu
NumPy             2.5.3
matplotlib        3.11.2
imageio           2.38.0


**期待する結果**（上の出力の読み方）: Python が `3.13.`、Gymnasium が `1.3.0`、Stable-Baselines3 が `2.9.0`、PyTorch が `2.14.` で始まっていれば、環境構築マニュアルのとおりに導入できています。後から導入した場合は、より新しい版の番号になっていることがあります。

## 2. 計算に使う装置を確かめる

PyTorch は、ニューラルネットワークの計算を CPU でも NVIDIA の GPU でも行えます。`torch.cuda.is_available()` は、GPU を使える状態なら `True`、使えなければ `False` を返す関数です。

この教科書では、後の章でも、次の1行で装置を選びます。GPU があれば `"cuda"`（GPU）を、無ければ `"cpu"` を選ぶ書き方です。GPU の有無で教材のコードを書き分ける必要はありません。

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("使う装置:", device)
if device == "cuda":
    print("GPU の名前:", torch.cuda.get_device_name(0))

使う装置: cpu


**期待する結果**（上の出力の読み方）: `使う装置: cpu` なら CPU で、`使う装置: cuda` なら GPU で計算します。GPU が無くても、この章は CPU だけで最後まで動きます。後の章でも、各章の冒頭に、使う装置と CPU での学習時間の目安を書きます。GPU 版の PyTorch を入れたのに `cpu` と表示される場合は、環境構築マニュアルの5-2節の期待する結果（`torch-` の後ろの `+cu126` のような印）を見直してください。

## 3. CartPole をでたらめに動かす

**CartPole** は、左右に動く台車の上に棒を立て、倒さないように台車を動かし続ける課題です。強化学習では、このような課題の世界を**環境**と呼び、環境の中で行動を選ぶ側を**エージェント**と呼びます。

ここでは、エージェントが何も考えず、毎回でたらめに「左へ押す」か「右へ押す」かを選んだらどうなるかを見ます。次のセルの `run_random_episode` 関数は、環境を最初の状態に戻し、棒が倒れるか一定の回数に達するまで、でたらめな行動を繰り返します。1回分の試行（最初から終わりまで）を**エピソード**と呼びます。

関数の中で使っている `reset()`（最初の状態に戻す）・`step()`（行動を1つ実行する）・`render()`（今の様子を絵にする）は、Gymnasium のすべての環境に共通する操作です。それぞれが返す値の意味は、第2章で詳しく扱います。ここでは、動かして絵が得られることだけを確かめます。

<!-- TODO（著者用）: 第2章を書いたら、上の「第2章」に節の番号と中身を添える -->

In [3]:
# でたらめな行動で1エピソードを走らせ、各時点の絵と結果を返す
def run_random_episode(env, seed):
    observation, info = env.reset(seed=seed)
    env.action_space.seed(seed)
    frames = [env.render()]
    total_reward = 0.0
    steps = 0
    while True:
        action = env.action_space.sample()
        observation, reward, terminated, truncated, info = env.step(action)
        frames.append(env.render())
        total_reward += reward
        steps += 1
        if terminated or truncated:
            return frames, total_reward, steps, terminated, truncated


env = gym.make("CartPole-v1", render_mode="rgb_array")
frames, total_reward, steps, terminated, truncated = run_random_episode(env, seed=0)
env.close()

print("続いた回数:", steps)
print("得た報酬の合計:", total_reward)
print("棒が倒れて終わったか:", terminated)
print("回数の上限で終わったか:", truncated)
print("絵の大きさ（高さ, 幅, 色）:", frames[0].shape)

続いた回数: 18
得た報酬の合計: 18.0
棒が倒れて終わったか: True
回数の上限で終わったか: False
絵の大きさ（高さ, 幅, 色）: (400, 600, 3)


**期待する結果**（上の出力の読み方）:

- **続いた回数**: 棒が倒れるまでに、台車を何回動かせたか。この seed では 18 回でした。著者が seed を 0〜99 に変えて100回試したところ、平均は約 25 回、最短は 9 回、最長は 93 回で、100回中87回は 10〜40 回で倒れました
- **得た報酬の合計**: CartPole では、棒が立っている間、1回動かすごとに報酬（ほうび）が 1 もらえます。そのため、続いた回数と同じ値になります。強化学習のエージェントは、この報酬の合計を大きくすることを目指して学習します
- **棒が倒れて終わったか / 回数の上限で終わったか**: エピソードの終わり方は2種類あります。棒が倒れた（または台車が外へ出た）場合は前者、上限の 500 回に達した場合は後者が `True` になります
- **絵の大きさ**: `render_mode="rgb_array"` を指定すると、`render()` は絵を数値の配列として返します。`(400, 600, 3)` は、高さ 400、幅 600 の画素が、それぞれ赤・緑・青の3つの値を持つという意味です

`seed=0` は、でたらめの出方を固定する指定です。同じ版のライブラリで同じ値を使えば、何度実行しても同じ結果になります（ライブラリの版が変わると、結果が変わることがあります）。値を変えると、続く回数も変わります。

なお、`render_mode="human"` を指定すると、別のウィンドウで動きを見ることもできます。この教科書では、Notebook の中で結果を残すため、絵を受け取って GIF にする方法を使います。

## 4. GIF にして見る

集めた絵を、`imageio` というライブラリで GIF アニメーションに保存します。保存先は、この Notebook と同じフォルダにある `figures` フォルダです。

In [4]:
from pathlib import Path

gif_path = Path("figures") / "ch00_random_cartpole.gif"
gif_path.parent.mkdir(exist_ok=True)
imageio.v3.imwrite(gif_path, frames, duration=100, loop=0)
print("保存した GIF:", gif_path.as_posix(), f"（{len(frames)} 枚の絵）")

保存した GIF: figures/ch00_random_cartpole.gif （19 枚の絵）


**期待する結果**（上の出力の読み方）: `figures/ch00_random_cartpole.gif` が保存され、枚数は「続いた回数」に最初の1枚を足した数（この seed では 19 枚）になります。

`duration=100` は、絵を1枚あたり 100 ミリ秒ずつ表示する（1秒あたり 10 枚）指定です。CartPole の中の時間は1回の行動あたり 0.02 秒なので、この GIF は実際の5倍ゆっくり再生されます。倒れていく様子を目で追いやすくするためです。`loop=0` は、アニメーションを繰り返し再生する指定です。

保存した GIF は、次のように表示されます。棒が少しずつ傾き、立て直せないまま倒れる様子が分かります。

![でたらめに動かした CartPole。棒が傾いていき、短い時間で倒れる](figures/ch00_random_cartpole.gif)

## 5. まとめ

この章では、次のことを確かめました。

| 確かめたこと | 使ったもの |
|---|---|
| ライブラリの版 | 各ライブラリの `__version__` |
| 計算に使う装置 | `torch.cuda.is_available()` と、装置を選ぶ1行 |
| 環境が動き、絵を出せること | `gym.make()`・`reset()`・`step()`・`render()` |
| 動きを GIF で残せること | `imageio.v3.imwrite()` |

でたらめに動かした台車では、棒はすぐに倒れてしまいました。次の第1章では、同じ CartPole を、Stable-Baselines3 というライブラリの学習アルゴリズムで数分だけ学習させます。学習の前と後で、棒の立ち方がどう変わるかを見比べます。

- 次に読むもの: 第1章 動かして楽しむ（準備中）

<!-- TODO（著者用）: 第1章ができたら ../ch01_first_training/ch01_first_training.ipynb へのリンクにする -->

## 出典

この章は、次の公式の文書（2026-10-01 確認）をもとに、著者の言葉でまとめたものです。逐語の転載ではありません。

- Farama Foundation, Gymnasium Documentation（CartPole の環境の説明）: https://gymnasium.farama.org/environments/classic_control/cart_pole/
- PyTorch, `torch.cuda.is_available` の定義（公式リポジトリの v2.14.1 の `torch/cuda/__init__.py`）: https://github.com/pytorch/pytorch/blob/v2.14.1/torch/cuda/__init__.py

4節の GIF の絵は、Gymnasium（MIT License）の CartPole の描画処理が出力したものです。この教科書のライセンスと、使っている第三者のソフトウェアの一覧は、リポジトリの [LICENSE.md](../../LICENSE.md) にあります。